In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

DATA_DIR = Path("../data/raw")
REPORT_DIR = Path("../reports")
REPORT_DIR.mkdir(parents=True, exist_ok=True)

train = pd.read_csv(DATA_DIR / "train.csv")
campaigns = pd.read_csv(DATA_DIR / "campaign_data.csv")
customers = pd.read_csv(DATA_DIR / "customer_demographics.csv")

print("Datasets loaded successfully!")
print("-" * 45)

print("Training data:", train.shape)
print("Campaign data:", campaigns.shape)
print("Customer demographics:", customers.shape)

Datasets loaded successfully!
---------------------------------------------
Training data: (78369, 5)
Campaign data: (28, 4)
Customer demographics: (760, 7)


In [2]:
# 1. Count records for each campaign
campaign_counts = (
    train.groupby("campaign_id")
    .size()
    .reset_index(name="training_records")
    .sort_values("training_records", ascending=False)
)

print("Training records per campaign:")
display(campaign_counts)


# 2. Count unique coupons in each campaign
campaign_coupon_counts = (
    train.groupby("campaign_id")
    .agg(
        training_records=("id", "count"),
        unique_coupons=("coupon_id", "nunique"),
        unique_customers=("customer_id", "nunique")
    )
    .reset_index()
    .sort_values("training_records", ascending=False)
)

print("\nCampaign and coupon summary:")
display(campaign_coupon_counts)


# 3. Overall campaign and coupon counts
print("\nOverall Summary")
print("-" * 40)
print("Unique campaigns:", train["campaign_id"].nunique())
print("Unique coupons:", train["coupon_id"].nunique())
print("Unique customers:", train["customer_id"].nunique())
print("Total training records:", len(train))

Training records per campaign:


,campaign_id,training_records
12,13,22606
7,8,22555
17,30,6467
13,26,6056
16,29,3895
8,9,3168
10,11,2782
11,12,2550
4,5,1827
9,10,1723



Campaign and coupon summary:


,campaign_id,training_records,unique_coupons,unique_customers
12,13,22606,207,1077
7,8,22555,208,1076
17,30,6467,178,361
13,26,6056,181,332
16,29,3895,33,118
8,9,3168,18,176
10,11,2782,13,214
11,12,2550,15,170
4,5,1827,11,166
9,10,1723,14,123



Overall Summary
----------------------------------------
Unique campaigns: 18
Unique coupons: 866
Unique customers: 1428
Total training records: 78369


In [3]:
# Calculate redemption statistics for each campaign

campaign_redemption = (
    train.groupby("campaign_id")
    .agg(
        total_records=("redemption_status", "count"),
        total_redemptions=("redemption_status", "sum"),
        unique_customers=("customer_id", "nunique"),
        unique_coupons=("coupon_id", "nunique")
    )
    .reset_index()
)

campaign_redemption["redemption_rate_pct"] = (
    campaign_redemption["total_redemptions"]
    / campaign_redemption["total_records"]
    * 100
)

campaign_redemption = campaign_redemption.sort_values(
    "redemption_rate_pct",
    ascending=False
)

display(campaign_redemption.round(2))

,campaign_id,total_records,total_redemptions,unique_customers,unique_coupons,redemption_rate_pct
5,6,65,1,65,1,1.54
12,13,22606,346,1077,207,1.53
7,8,22555,199,1076,208,0.88
13,26,6056,52,332,181,0.86
8,9,3168,23,176,18,0.73
3,4,972,7,81,12,0.72
0,1,143,1,13,11,0.70
1,2,768,5,48,16,0.65
17,30,6467,39,361,178,0.60
2,3,408,2,12,34,0.49


In [4]:
# Inspect campaign columns
print("CAMPAIGN DATA COLUMNS")
print("-" * 45)
print(campaigns.columns.tolist())

# Inspect training columns
print("\nTRAINING DATA COLUMNS")
print("-" * 45)
print(train.columns.tolist())

# Check whether customer-level campaign assignment exists
print("\nCUSTOMER CAMPAIGN REPRESENTATION")
print("-" * 45)

print(
    "Campaign ID available in training:",
    "campaign_id" in train.columns
)

print(
    "Campaign ID available in customer demographics:",
    "campaign_id" in customers.columns
)

print(
    "Unique customer IDs in training:",
    train["customer_id"].nunique()
)

print(
    "Unique customer IDs in demographics:",
    customers["customer_id"].nunique()
)

# Check how many training records match customer demographics
matched = train["customer_id"].isin(
    customers["customer_id"]
).sum()

print(
    "Training records matching customer demographics:",
    matched,
    "out of",
    len(train)
)

print(
    "Percentage matched:",
    round(matched / len(train) * 100, 2),
    "%"
)

CAMPAIGN DATA COLUMNS
---------------------------------------------
['campaign_id', 'campaign_type', 'start_date', 'end_date']

TRAINING DATA COLUMNS
---------------------------------------------
['id', 'campaign_id', 'coupon_id', 'customer_id', 'redemption_status']

CUSTOMER CAMPAIGN REPRESENTATION
---------------------------------------------
Campaign ID available in training: True
Campaign ID available in customer demographics: False
Unique customer IDs in training: 1428
Unique customer IDs in demographics: 760
Training records matching customer demographics: 43661 out of 78369
Percentage matched: 55.71 %


In [5]:
# Save campaign analysis outputs

campaign_counts.to_csv(
    REPORT_DIR / "day09_campaign_counts.csv",
    index=False
)

campaign_redemption.to_csv(
    REPORT_DIR / "day09_campaign_redemption.csv",
    index=False
)

print("Day 9 analysis files saved successfully!")
print("\nFiles created:")
print("1. day09_campaign_counts.csv")
print("2. day09_campaign_redemption.csv")

Day 9 analysis files saved successfully!

Files created:
1. day09_campaign_counts.csv
2. day09_campaign_redemption.csv
